In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
test_path = "C:\\Users\\rakib\\Downloads\\mental_health_combined_test.csv"
train_path = "C:\\Users\\rakib\\Downloads\\mental_heath_unbanlanced.csv"

In [3]:
train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

In [4]:
def load_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    print("Train shape:", train_df.shape)
    print("Test  shape:", test_df.shape)
    print("\nColumns:", train_df.columns.tolist())
    print("\nFirst rows:\n", train_df.head(3))
    return train_df, test_df

# ── Adjust column names if needed ────────────────────────────
TEXT_COL  = "text"      # change if your column is named differently
LABEL_COL = "status"     # change if your column is named differently

train_df, test_df = load_data(train_path, test_path)

Train shape: (49612, 3)
Test  shape: (992, 2)

Columns: ['Unique_ID', 'text', 'status']

First rows:
    Unique_ID                                               text   status
0        0.0                                         oh my gosh  Anxiety
1        1.0  trouble sleeping, confused mind, restless hear...  Anxiety
2        2.0  All wrong, back off dear, forward doubt. Stay ...  Anxiety


In [5]:
# SECTION 3: EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================
 
def run_eda(df, label_col=LABEL_COL, text_col=TEXT_COL):
    print("\n── CLASS DISTRIBUTION ──────────────────────────────")
    print(df[label_col].value_counts())
 
    # Plot class distribution
    plt.figure(figsize=(10, 5))
    ax = df[label_col].value_counts().plot(kind="bar", color="steelblue", edgecolor="white")
    plt.title("Class Distribution")
    plt.xlabel("Mental Health Category")
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")
    for p in ax.patches:
        ax.annotate(str(p.get_height()), (p.get_x() + 0.1, p.get_height() + 5))
    plt.tight_layout()
    plt.savefig("eda_class_distribution.png", dpi=150)
    plt.close()
    print("Saved: eda_class_distribution.png")
 
    # Text length analysis
    df["text_length"] = df[text_col].astype(str).apply(lambda x: len(x.split()))
    print("\n── TEXT LENGTH STATS ───────────────────────────────")
    print(df.groupby(label_col)["text_length"].describe())
 
    plt.figure(figsize=(12, 5))
    for label in df[label_col].unique():
        subset = df[df[label_col] == label]["text_length"]
        subset.plot(kind="kde", label=label)
    plt.title("Text Length Distribution per Class")
    plt.xlabel("Word Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig("eda_text_length.png", dpi=150)
    plt.close()
    print("Saved: eda_text_length.png")
 
    # Null check
    print("\n── NULL VALUES ─────────────────────────────────────")
    print(df[[text_col, label_col]].isnull().sum())
 
run_eda(train_df)
 
# Word clouds per class
try:
    from wordcloud import WordCloud
 
    def plot_wordclouds(df, label_col=LABEL_COL, text_col=TEXT_COL):
        labels = df[label_col].unique()
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.flatten()
        for i, label in enumerate(labels):
            text = " ".join(df[df[label_col] == label][text_col].astype(str).tolist())
            wc = WordCloud(width=400, height=300, background_color="white",
                           max_words=80, colormap="viridis").generate(text)
            axes[i].imshow(wc, interpolation="bilinear")
            axes[i].set_title(label, fontsize=13, fontweight="bold")
            axes[i].axis("off")
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")
        plt.suptitle("Word Clouds by Mental Health Category", fontsize=16, y=1.01)
        plt.tight_layout()
        plt.savefig("eda_wordclouds.png", dpi=150)
        plt.close()
        print("Saved: eda_wordclouds.png")
 
    plot_wordclouds(train_df)
except ImportError:
    print("wordcloud not installed — skipping word cloud plot")



── CLASS DISTRIBUTION ──────────────────────────────
status
Normal        18391
Depression    14506
Suicidal      11212
Anxiety        5503
Name: count, dtype: int64
Saved: eda_class_distribution.png

── TEXT LENGTH STATS ───────────────────────────────
              count        mean         std  min   25%   50%    75%     max
status                                                                     
Anxiety      5503.0  123.395784  144.001970  2.0  49.0  94.0  157.0  2809.0
Depression  14506.0  106.116090  105.196009  1.0  47.0  87.0  148.0  4426.0
Normal      18391.0   23.500788   72.544228  1.0   6.0  11.0   22.0  6353.0
Suicidal    11212.0  109.751784  158.192997  1.0  38.0  80.0  143.0  9684.0
Saved: eda_text_length.png

── NULL VALUES ─────────────────────────────────────
text      0
status    0
dtype: int64
Saved: eda_wordclouds.png


In [6]:
# SECTION 4: TEXT PREPROCESSING
# ==============================================================
%pip install nltk
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
 
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
 
lemmatizer  = WordNetLemmatizer()
stop_words  = set(stopwords.words("english"))
 
# Keep negation words — important for mental health text
KEEP_WORDS = {"not", "no", "never", "none", "nor", "cannot", "couldn't",
              "don't", "doesn't", "didn't", "won't", "wouldn't", "shouldn't"}
stop_words -= KEEP_WORDS
 
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)          # remove URLs
    text = re.sub(r"@\w+|#\w+", "", text)               # remove mentions/hashtags
    text = re.sub(r"[^a-z\s']", " ", text)              # keep letters + apostrophe
    text = re.sub(r"\s+", " ", text).strip()            # collapse whitespace
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words or t in KEEP_WORDS]
    return " ".join(tokens)
 
print("\nPreprocessing texts...")
train_df["clean_text"] = train_df[TEXT_COL].astype(str).apply(clean_text)
test_df["clean_text"]  = test_df[TEXT_COL].astype(str).apply(clean_text)
print("Sample cleaned text:")
print(train_df["clean_text"].iloc[0])


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.

Preprocessing texts...
Sample cleaned text:
oh gosh


In [7]:
# SECTION 5: LABEL ENCODING
# ==============================================================
from sklearn.preprocessing import LabelEncoder
 
le = LabelEncoder()
train_df["label_enc"] = le.fit_transform(train_df[LABEL_COL])
test_df["label_enc"]  = le.transform(test_df[LABEL_COL])
 
print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")
 
NUM_CLASSES = len(le.classes_)
print(f"\nTotal classes: {NUM_CLASSES}")
 
X_train = train_df["clean_text"].values
y_train = train_df["label_enc"].values
X_test  = test_df["clean_text"].values
y_test  = test_df["label_enc"].values
 


Label mapping:
  0 → Anxiety
  1 → Depression
  2 → Normal
  3 → Suicidal

Total classes: 4


In [8]:
# SECTION 6: HANDLE CLASS IMBALANCE
# ==============================================================
from collections import Counter
 
print("\nClass counts before balancing:", Counter(y_train))
 
# Compute class weights for models that support it
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", {le.classes_[k]: round(v, 3) for k, v in class_weight_dict.items()})
 


Class counts before balancing: Counter({np.int64(2): 18391, np.int64(1): 14506, np.int64(3): 11212, np.int64(0): 5503})
Class weights: {'Anxiety': np.float64(2.254), 'Depression': np.float64(0.855), 'Normal': np.float64(0.674), 'Suicidal': np.float64(1.106)}


In [9]:
# SECTION 7: FEATURE EXTRACTION — TF-IDF
# ==============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
 
print("\nFitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),    # unigrams + bigrams
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode",
)
 
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")



Fitting TF-IDF vectorizer...
TF-IDF matrix shape: (49612, 20000)


In [10]:
 # SECTION 8: CLASSICAL ML MODELS
# ==============================================================
from sklearn.linear_model   import LogisticRegression
from sklearn.naive_bayes    import MultinomialNB
from sklearn.ensemble       import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm            import LinearSVC
from sklearn.metrics        import (classification_report, confusion_matrix,
                                    accuracy_score, f1_score)
import time
 
def evaluate_model(name, y_true, y_pred, classes):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"  Accuracy : {acc:.4f}  |  Macro-F1 : {f1:.4f}")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred, target_names=classes))
 
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.title(f"Confusion Matrix — {name}")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    fname = f"cm_{name.replace(' ', '_').lower()}.png"
    plt.savefig(fname, dpi=130)
    plt.close()
    return acc, f1
 
results = {}

In [11]:
# ── 8.1 Logistic Regression ──────────────────────────────────
print("\nTraining Logistic Regression...")
t0 = time.time()
lr_model = LogisticRegression(
    max_iter=1000, C=1.0, solver="lbfgs",
    class_weight="balanced", random_state=42
)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
print(f"Training time: {time.time()-t0:.1f}s")
acc, f1 = evaluate_model("Logistic Regression", y_test, lr_preds, le.classes_)
results["Logistic Regression"] = {"accuracy": acc, "macro_f1": f1}


Training Logistic Regression...
Training time: 2.0s

  Logistic Regression
  Accuracy : 0.7530  |  Macro-F1 : 0.7503
              precision    recall  f1-score   support

     Anxiety       0.79      0.79      0.79       248
  Depression       0.67      0.58      0.62       248
      Normal       0.82      0.86      0.84       248
    Suicidal       0.72      0.79      0.75       248

    accuracy                           0.75       992
   macro avg       0.75      0.75      0.75       992
weighted avg       0.75      0.75      0.75       992



In [12]:
# ── 8.2 Multinomial Naive Bayes ──────────────────────────────
print("\nTraining Naive Bayes...")
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
acc, f1 = evaluate_model("Naive Bayes", y_test, nb_preds, le.classes_)
results["Naive Bayes"] = {"accuracy": acc, "macro_f1": f1}


Training Naive Bayes...

  Naive Bayes
  Accuracy : 0.6683  |  Macro-F1 : 0.6695
              precision    recall  f1-score   support

     Anxiety       0.91      0.48      0.63       248
  Depression       0.48      0.67      0.56       248
      Normal       0.74      0.85      0.80       248
    Suicidal       0.73      0.67      0.70       248

    accuracy                           0.67       992
   macro avg       0.72      0.67      0.67       992
weighted avg       0.72      0.67      0.67       992



In [13]:
# ── 8.3 Linear SVM ───────────────────────────────────────────
print("\nTraining Linear SVM...")
svm_model = LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)
acc, f1 = evaluate_model("Linear SVM", y_test, svm_preds, le.classes_)
results["Linear SVM"] = {"accuracy": acc, "macro_f1": f1}


Training Linear SVM...

  Linear SVM
  Accuracy : 0.7812  |  Macro-F1 : 0.7735
              precision    recall  f1-score   support

     Anxiety       0.83      0.70      0.76       248
  Depression       0.70      0.56      0.62       248
      Normal       0.81      0.94      0.87       248
    Suicidal       0.77      0.93      0.84       248

    accuracy                           0.78       992
   macro avg       0.78      0.78      0.77       992
weighted avg       0.78      0.78      0.77       992



In [14]:
# ── 8.4 Random Forest ────────────────────────────────────────
print("\nTraining Random Forest (this may take a few minutes)...")
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=None, class_weight="balanced",
    n_jobs=-1, random_state=42
)
rf_model.fit(X_train_tfidf, y_train)
rf_preds = rf_model.predict(X_test_tfidf)
acc, f1 = evaluate_model("Random Forest", y_test, rf_preds, le.classes_)
results["Random Forest"] = {"accuracy": acc, "macro_f1": f1}


Training Random Forest (this may take a few minutes)...

  Random Forest
  Accuracy : 0.8004  |  Macro-F1 : 0.7863
              precision    recall  f1-score   support

     Anxiety       0.98      0.46      0.63       248
  Depression       0.64      0.74      0.69       248
      Normal       0.85      1.00      0.92       248
    Suicidal       0.84      1.00      0.91       248

    accuracy                           0.80       992
   macro avg       0.83      0.80      0.79       992
weighted avg       0.83      0.80      0.79       992



In [15]:
# ── 8.5 XGBoost ──────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    print("\nTraining XGBoost...")
    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train_tfidf, y_train)
    xgb_preds = xgb_model.predict(X_test_tfidf)
    acc, f1 = evaluate_model("XGBoost", y_test, xgb_preds, le.classes_)
    results["XGBoost"] = {"accuracy": acc, "macro_f1": f1}
except ImportError:
    print("XGBoost not installed — skipping")


Training XGBoost...

  XGBoost
  Accuracy : 0.7510  |  Macro-F1 : 0.7499
              precision    recall  f1-score   support

     Anxiety       0.91      0.68      0.78       248
  Depression       0.64      0.65      0.64       248
      Normal       0.76      0.92      0.83       248
    Suicidal       0.74      0.76      0.75       248

    accuracy                           0.75       992
   macro avg       0.76      0.75      0.75       992
weighted avg       0.76      0.75      0.75       992



In [16]:
# ── 8.6 LightGBM ─────────────────────────────────────────────
try:
    from lightgbm import LGBMClassifier
    print("\nTraining LightGBM...")
    lgbm_model = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, class_weight="balanced",
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgbm_model.fit(X_train_tfidf, y_train)
    lgbm_preds = lgbm_model.predict(X_test_tfidf)
    acc, f1 = evaluate_model("LightGBM", y_test, lgbm_preds, le.classes_)
    results["LightGBM"] = {"accuracy": acc, "macro_f1": f1}
except ImportError:
    print("LightGBM not installed — skipping")


Training LightGBM...

  LightGBM
  Accuracy : 0.7671  |  Macro-F1 : 0.7630
              precision    recall  f1-score   support

     Anxiety       0.79      0.79      0.79       248
  Depression       0.72      0.57      0.64       248
      Normal       0.83      0.87      0.85       248
    Suicidal       0.72      0.85      0.78       248

    accuracy                           0.77       992
   macro avg       0.77      0.77      0.76       992
weighted avg       0.77      0.77      0.76       992



In [17]:
# SECTION 9: RESULTS COMPARISON TABLE
# ==============================================================
print("\n\n" + "="*55)
print("  MODEL COMPARISON SUMMARY")
print("="*55)
results_df = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
print(results_df.to_string())
 
plt.figure(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.35
plt.bar(x - w/2, results_df["accuracy"],  w, label="Accuracy",  color="steelblue")
plt.bar(x + w/2, results_df["macro_f1"],  w, label="Macro-F1",  color="coral")
plt.xticks(x, results_df.index, rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.title("Classical ML Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.close()
print("\nSaved: model_comparison.png")



  MODEL COMPARISON SUMMARY
                     accuracy  macro_f1
Random Forest        0.800403  0.786346
Linear SVM           0.781250  0.773510
LightGBM             0.767137  0.762987
Logistic Regression  0.753024  0.750263
XGBoost              0.751008  0.749923
Naive Bayes          0.668347  0.669509

Saved: model_comparison.png


In [18]:
# SECTION 10: SHALLOW DEEP LEARNING — BiLSTM + Attention
# ==============================================================
print("\n\n── SECTION 10: SHALLOW DL (BiLSTM) ────────────────────")
 
%pip install tensorflow
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models        import Model
from tensorflow.keras.layers        import (Input, Embedding, Bidirectional,
                                             LSTM, Dense, Dropout,
                                             GlobalAveragePooling1D,
                                             LayerNormalization, MultiHeadAttention)
from tensorflow.keras.callbacks     import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers    import Adam
from tensorflow.keras.utils         import to_categorical
 
tf.random.set_seed(42)



── SECTION 10: SHALLOW DL (BiLSTM) ────────────────────
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [22]:
# Hyperparameters
VOCAB_SIZE  = 30000
MAX_LEN     = 200    # max tokens per sample
EMBED_DIM   = 128
LSTM_UNITS  = 128
DROPOUT     = 0.3
BATCH_SIZE  = 32
EPOCHS      = 20
 
# ── 10.1 Tokenize ─────────────────────────────────────────────
tokenizer_dl = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer_dl.fit_on_texts(X_train)
 
X_train_seq = pad_sequences(tokenizer_dl.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq  = pad_sequences(tokenizer_dl.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding="post", truncating="post")
 
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=NUM_CLASSES)
 
print(f"X_train_seq shape: {X_train_seq.shape}")

X_train_seq shape: (49612, 200)


In [23]:
# ── 10.2 Build BiLSTM Model ───────────────────────────────────
def build_bilstm_model(vocab_size, embed_dim, max_len, lstm_units,
                        num_classes, dropout_rate):
    inputs = Input(shape=(max_len,))
    x = Embedding(vocab_size, embed_dim, mask_zero=True)(inputs)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(lstm_units, return_sequences=True, use_cudnn=False))(x)
    x = LayerNormalization()(x)
    # Self-attention on LSTM output
    attn_out = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = x + attn_out
    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs, outputs)
    return model

bilstm_model = build_bilstm_model(
    VOCAB_SIZE, EMBED_DIM, MAX_LEN, LSTM_UNITS, NUM_CLASSES, DROPOUT
)
bilstm_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
bilstm_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 200, 128)  │  3,840,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 200, 128)  │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 200)       │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 200, 256)  │    263,168 │ dropout_8[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 200, 256)  │        512 │ bidirectional_2[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 200, 256)  │    131,712 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 200, 256)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ add_2[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 256)       │          0 │ dense_6[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │     32,896 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 128)       │          0 │ dense_7[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 4)         │        516 │ dropout_11[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,334,596 (16.54 MB)

 Trainable params: 4,334,596 (16.54 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
# ── 10.3 Train ────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]
 
print("\nTraining BiLSTM...")
history = bilstm_model.fit(
    X_train_seq, y_train_cat,
    validation_split=0.15,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)


Training BiLSTM...
Epoch 1/20
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 387s 291ms/step - accuracy: 0.7419 - loss: 0.6350 - val_accuracy: 0.6200 - val_loss: 0.9625 - learning_rate: 0.0010
Epoch 2/20
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 390s 296ms/step - accuracy: 0.8099 - loss: 0.4562 - val_accuracy: 0.6321 - val_loss: 0.9956 - learning_rate: 0.0010
Epoch 3/20
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 389s 295ms/step - accuracy: 0.8431 - loss: 0.3773 - val_accuracy: 0.6051 - val_loss: 1.2306 - learning_rate: 0.0010
Epoch 4/20
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 395s 300ms/step - accuracy: 0.8837 - loss: 0.2792 - val_accuracy: 0.6043 - val_loss: 1.6123 - learning_rate: 5.0000e-04
Epoch 5/20
1318/1318 ━━━━━━━━━━━━━━━━━━━━ 401s 304ms/step - accuracy: 0.9084 - loss: 0.2260 - val_accuracy: 0.5830 - val_loss: 1.9398 - learning_rate: 5.0000e-04


In [26]:
# ── 10.4 Evaluate ─────────────────────────────────────────────
bilstm_preds = np.argmax(bilstm_model.predict(X_test_seq, batch_size=64), axis=1)
acc, f1 = evaluate_model("BiLSTM + Attention", y_test, bilstm_preds, le.classes_)
results["BiLSTM + Attention"] = {"accuracy": acc, "macro_f1": f1}
 
# ── 10.5 Plot training curves ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"],     label="Train")
ax1.plot(history.history["val_accuracy"], label="Val")
ax1.set_title("Accuracy"); ax1.legend()
ax2.plot(history.history["loss"],         label="Train")
ax2.plot(history.history["val_loss"],     label="Val")
ax2.set_title("Loss"); ax2.legend()
plt.suptitle("BiLSTM Training History")
plt.tight_layout()
plt.savefig("bilstm_training_curves.png", dpi=130)
plt.close()
print("Saved: bilstm_training_curves.png")

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step

  BiLSTM + Attention
  Accuracy : 0.6240  |  Macro-F1 : 0.6244
              precision    recall  f1-score   support

     Anxiety       0.85      0.59      0.70       248
  Depression       0.51      0.57      0.54       248
      Normal       0.60      0.80      0.69       248
    Suicidal       0.62      0.54      0.58       248

    accuracy                           0.62       992
   macro avg       0.65      0.62      0.62       992
weighted avg       0.65      0.62      0.62       992

Saved: bilstm_training_curves.png


In [27]:
# SECTION 11: SAVE ARTIFACTS
# ==============================================================
import joblib
 
joblib.dump(tfidf,    "tfidf_vectorizer.pkl")
joblib.dump(lr_model, "logistic_regression_model.pkl")
joblib.dump(le,       "label_encoder.pkl")
bilstm_model.save("bilstm_model.keras")
 
print("\n── All artifacts saved ──────────────────────────────")
print("  tfidf_vectorizer.pkl")
print("  logistic_regression_model.pkl")
print("  label_encoder.pkl")
print("  bilstm_model.keras")


── All artifacts saved ──────────────────────────────
  tfidf_vectorizer.pkl
  logistic_regression_model.pkl
  label_encoder.pkl
  bilstm_model.keras


In [28]:
# SECTION 12: INFERENCE DEMO
# ==============================================================
def predict_mental_health(text: str, model=lr_model, vectorizer=tfidf,
                           label_enc=le, use_dl=False, dl_model=None,
                           dl_tokenizer=None, max_len=MAX_LEN) -> dict:
    """Predict mental health category for a given text."""
    cleaned = clean_text(text)
    if use_dl and dl_model is not None:
        seq = pad_sequences(dl_tokenizer.texts_to_sequences([cleaned]),
                            maxlen=max_len, padding="post")
        probs = dl_model.predict(seq, verbose=0)[0]
        pred_idx = np.argmax(probs)
        return {
            "text":       text[:80],
            "prediction": label_enc.classes_[pred_idx],
            "confidence": f"{probs[pred_idx]*100:.1f}%",
            "all_probs":  {label_enc.classes_[i]: f"{p*100:.1f}%"
                           for i, p in enumerate(probs)}
        }
    else:
        vec = vectorizer.transform([cleaned])
        pred_idx = model.predict(vec)[0]
        proba    = model.predict_proba(vec)[0] if hasattr(model, "predict_proba") else None
        return {
            "text":       text[:80],
            "prediction": label_enc.classes_[pred_idx],
            "confidence": f"{max(proba)*100:.1f}%" if proba is not None else "N/A",
        }
 
# Demo
test_texts = [
    "I feel so hopeless and can't stop crying every day",
    "I'm always worried about everything, my heart races constantly",
    "Having a great day with friends, feeling happy",
    "I keep thinking about ending it all, nobody would care",
    "My mood swings so fast, one minute I'm on top of the world",
]
 
print("\n── INFERENCE DEMO ──────────────────────────────────────")
for txt in test_texts:
    result = predict_mental_health(txt)
    print(f"\n  Text       : {result['text']}")
    print(f"  Prediction : {result['prediction']}")
    print(f"  Confidence : {result['confidence']}")
 
print("\n\n✅ FULL ML PIPELINE COMPLETE")
print("Next step: Run mental_health_bert_pipeline.py for BERT fine-tuning")


── INFERENCE DEMO ──────────────────────────────────────

  Text       : I feel so hopeless and can't stop crying every day
  Prediction : Depression
  Confidence : 48.0%

  Text       : I'm always worried about everything, my heart races constantly
  Prediction : Anxiety
  Confidence : 90.3%

  Text       : Having a great day with friends, feeling happy
  Prediction : Normal
  Confidence : 65.7%

  Text       : I keep thinking about ending it all, nobody would care
  Prediction : Suicidal
  Confidence : 71.0%

  Text       : My mood swings so fast, one minute I'm on top of the world
  Prediction : Normal
  Confidence : 74.8%


✅ FULL ML PIPELINE COMPLETE
Next step: Run mental_health_bert_pipeline.py for BERT fine-tuning


In [42]:
# MENTAL HEALTH TEXT CLASSIFICATION — BERT FINE-TUNING PIPELINE

MODEL_NAME  = "bert-base-uncased"
TRAIN_PATH = "mental_heath_unbanlanced.csv" # Corrected typo in filename
TEST_PATH = "mental_health_combined_test.csv"
OUTPUT_DIR  = "./bert_checkpoints"

TEXT_COL    = "text"
LABEL_COL   = "label"


In [55]:
# ── Training hyperparameters ─────────────────────────────────
MAX_LEN     = 128     
BATCH_SIZE  = 100      
EPOCHS      = 3
LR          = 2e-5     # standard for BERT fine-tuning
WARMUP_RATIO= 0.1
WEIGHT_DECAY= 0.01
SEED        = 42

In [56]:
import os, re, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
 
import torch
from torch.utils.data       import Dataset, DataLoader
from torch.optim            import AdamW
from torch.cuda.amp         import autocast, GradScaler   # mixed precision
 
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    logging as hf_logging,
)
hf_logging.set_verbosity_error()
 
from sklearn.preprocessing  import LabelEncoder
from sklearn.metrics        import (classification_report, confusion_matrix,
                                    accuracy_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

In [57]:
# ── Device setup ─────────────────────────────────────────────
SEED = 42  # Define SEED if not already set
OUTPUT_DIR = "./bert_checkpoints"  # Define OUTPUT_DIR if not already set

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

Using device: cpu


In [58]:
# SECTION 3: LOAD & PREPROCESS DATA
# ==============================================================
 
def basic_clean(text: str) -> str:
    """Light cleaning — BERT tokenizer handles most normalization."""
    text = str(text).strip()
    text = re.sub(r"http\S+|www\S+", "[URL]", text)    # replace URLs
    text = re.sub(r"@\w+", "[USER]", text)             # replace @mentions
    text = re.sub(r"\s+", " ", text)
    return text
 
print("\nLoading data...")
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
 
train_df[TEXT_COL] = train_df[TEXT_COL].apply(basic_clean)
test_df[TEXT_COL]  = test_df[TEXT_COL].apply(basic_clean)


Loading data...
Train: 49612 rows | Test: 992 rows


In [59]:
# ── Label encoding ────────────────────────────────────────────
LABEL_COL = "status"  # Correct column name based on dataframe
le = LabelEncoder()
train_df["label_enc"] = le.fit_transform(train_df[LABEL_COL])
test_df["label_enc"]  = le.transform(test_df[LABEL_COL])

NUM_CLASSES = len(le.classes_)
id2label = {i: c for i, c in enumerate(le.classes_)}
label2id = {c: i for i, c in enumerate(le.classes_)}

print(f"\nClasses ({NUM_CLASSES}):")
for k, v in id2label.items():
    print(f"  {k} → {v}")


Classes (4):
  0 → Anxiety
  1 → Depression
  2 → Normal
  3 → Suicidal


In [60]:
# ── Class weights for imbalanced data ─────────────────────────
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(train_df["label_enc"].values),
    y=train_df["label_enc"].values
)
class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("\nClass weights:", {id2label[i]: round(w, 3) for i, w in enumerate(class_weights)})


Class weights: {'Anxiety': np.float64(2.254), 'Depression': np.float64(0.855), 'Normal': np.float64(0.674), 'Suicidal': np.float64(1.106)}


In [61]:
# SECTION 4: PYTORCH DATASET
# ==============================================================
 
class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt",
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "token_type_ids" : encoding.get("token_type_ids",
                               torch.zeros(self.max_len, dtype=torch.long)).squeeze(),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }
 

In [62]:
# SECTION 5: LOAD TOKENIZER & MODEL
# ==============================================================
 
print(f"\nLoading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
 
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = NUM_CLASSES,
    id2label   = id2label,
    label2id   = label2id,
    ignore_mismatched_sizes = True,
)
model = model.to(DEVICE)
 
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")


Loading tokenizer and model: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4904.81it/s]

Model parameters: 109,485,316


In [63]:
# SECTION 6: DATALOADERS
# ==============================================================
 
train_texts  = train_df[TEXT_COL].tolist()
train_labels = train_df["label_enc"].tolist()
test_texts   = test_df[TEXT_COL].tolist()
test_labels  = test_df["label_enc"].tolist()

# Split a small validation set from training
from sklearn.model_selection import train_test_split
tr_texts, val_texts, tr_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, stratify=train_labels, random_state=SEED
)
 
train_dataset = MentalHealthDataset(tr_texts,   tr_labels,   tokenizer, MAX_LEN)
val_dataset   = MentalHealthDataset(val_texts,  val_labels,  tokenizer, MAX_LEN)
test_dataset  = MentalHealthDataset(test_texts, test_labels, tokenizer, MAX_LEN)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=0, pin_memory=True)
 
print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
 


Train batches : 447
Val   batches : 25
Test  batches : 5


In [64]:
# SECTION 7: OPTIMIZER & SCHEDULER
# ==============================================================
 
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
 
# Use different LR for classifier head vs pre-trained layers
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay) and "classifier" not in n],
        "weight_decay": WEIGHT_DECAY,
        "lr": LR,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay) and "classifier" not in n],
        "weight_decay": 0.0,
        "lr": LR,
    },
    {
        "params": [p for n, p in model.named_parameters() if "classifier" in n],
        "weight_decay": WEIGHT_DECAY,
        "lr": LR * 10,     # higher LR for the classification head
    },
]
 
optimizer  = AdamW(optimizer_grouped_params)
scheduler  = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = GradScaler()   # for mixed-precision training
 
criterion = torch.nn.CrossEntropyLoss(weight=class_weight_tensor)
 

In [65]:
# SECTION 8: TRAINING & VALIDATION FUNCTIONS
# ==============================================================
 
def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
 
    for step, batch in enumerate(loader):
        input_ids  = batch["input_ids"].to(device)
        attn_mask  = batch["attention_mask"].to(device)
        tok_ids    = batch["token_type_ids"].to(device)
        labels     = batch["labels"].to(device)
 
        optimizer.zero_grad()
 
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attn_mask,
                            token_type_ids=tok_ids)
            loss    = criterion(outputs.logits, labels)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
 
        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
 
        if (step + 1) % 50 == 0:
            print(f"    step {step+1}/{len(loader)} | loss: {loss.item():.4f}")
 
    return total_loss / len(loader), correct / total
 
 
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
 
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            tok_ids   = batch["token_type_ids"].to(device)
            labels    = batch["labels"].to(device)
 
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attn_mask,
                                token_type_ids=tok_ids)
                loss    = criterion(outputs.logits, labels)
 
            total_loss  += loss.item()
            preds        = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds   .extend(preds)
            all_labels  .extend(labels.cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)
 

In [66]:
# SECTION 9: TRAINING LOOP
# ==============================================================
 
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
           "val_f1": [], "lr": []}
 
best_val_f1    = 0.0
best_model_path = os.path.join(OUTPUT_DIR, "best_model")
 
print("\n" + "="*60)
print(f"  Starting BERT fine-tuning: {MODEL_NAME}")
print(f"  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | MaxLen: {MAX_LEN} | LR: {LR}")
print("="*60)
 
for epoch in range(1, EPOCHS + 1):
    t_start = time.time()
    print(f"\n── Epoch {epoch}/{EPOCHS} ──────────────────────────────")
 
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, criterion, DEVICE
    )
    val_loss, val_acc, val_f1, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )
 
    elapsed = time.time() - t_start
    current_lr = scheduler.get_last_lr()[0]
 
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}")
    print(f"  Val   — loss: {val_loss:.4f}   | acc: {val_acc:.4f} | macro-F1: {val_f1:.4f}")
    print(f"  Time  : {elapsed:.0f}s | LR: {current_lr:.2e}")
 
    history["train_loss"].append(train_loss)
    history["val_loss"]  .append(val_loss)
    history["train_acc"] .append(train_acc)
    history["val_acc"]   .append(val_acc)
    history["val_f1"]    .append(val_f1)
    history["lr"]        .append(current_lr)
 
    # ── Save best checkpoint ──────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"  ✅ Best model saved (val macro-F1={best_val_f1:.4f})")
 


  Starting BERT fine-tuning: bert-base-uncased
  Epochs: 3 | Batch: 100 | MaxLen: 128 | LR: 2e-05

── Epoch 1/3 ──────────────────────────────
    step 50/447 | loss: 1.1663
    step 100/447 | loss: 0.8004
    step 150/447 | loss: 0.6210
    step 200/447 | loss: 0.4681
    step 250/447 | loss: 0.4792
    step 300/447 | loss: 0.4887
    step 350/447 | loss: 0.6062
    step 400/447 | loss: 0.4348
  Train — loss: 0.6564 | acc: 0.7239
  Val   — loss: 0.4374   | acc: 0.8251 | macro-F1: 0.8115
  Time  : 35956s | LR: 1.48e-05


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]


  ✅ Best model saved (val macro-F1=0.8115)

── Epoch 2/3 ──────────────────────────────
    step 50/447 | loss: 0.3041
    step 100/447 | loss: 0.3945
    step 150/447 | loss: 0.4962
    step 200/447 | loss: 0.2911
    step 250/447 | loss: 0.3586
    step 300/447 | loss: 0.5040
    step 350/447 | loss: 0.3323
    step 400/447 | loss: 0.2541
  Train — loss: 0.3736 | acc: 0.8462
  Val   — loss: 0.4099   | acc: 0.8345 | macro-F1: 0.8238
  Time  : 39252s | LR: 7.41e-06


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]


  ✅ Best model saved (val macro-F1=0.8238)

── Epoch 3/3 ──────────────────────────────
    step 50/447 | loss: 0.2605
    step 100/447 | loss: 0.3007
    step 150/447 | loss: 0.1552
    step 200/447 | loss: 0.3054
    step 250/447 | loss: 0.2525
    step 300/447 | loss: 0.3289
    step 350/447 | loss: 0.2544
    step 400/447 | loss: 0.2034
  Train — loss: 0.2919 | acc: 0.8789
  Val   — loss: 0.4200   | acc: 0.8356 | macro-F1: 0.8233
  Time  : 36415s | LR: 0.00e+00


In [67]:
# SECTION 10: TEST EVALUATION WITH BEST MODEL
# ==============================================================
 
print("\n\nLoading best checkpoint for final test evaluation...")
best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
best_model = best_model.to(DEVICE)
 
_, test_acc, test_f1, test_preds, test_true = evaluate(
    best_model, test_loader, criterion, DEVICE
)
 
print(f"\n{'='*60}")
print(f"  FINAL TEST RESULTS — {MODEL_NAME}")
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  Macro-F1  : {test_f1:.4f}")
print(f"{'='*60}")
print("\nPer-class report:")
print(classification_report(test_true, test_preds, target_names=le.classes_))



Loading best checkpoint for final test evaluation...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3614.37it/s]



  FINAL TEST RESULTS — bert-base-uncased
  Accuracy  : 0.8538
  Macro-F1  : 0.8521

Per-class report:
              precision    recall  f1-score   support

     Anxiety       0.82      0.88      0.85       248
  Depression       0.80      0.71      0.75       248
      Normal       0.91      0.90      0.91       248
    Suicidal       0.87      0.93      0.90       248

    accuracy                           0.85       992
   macro avg       0.85      0.85      0.85       992
weighted avg       0.85      0.85      0.85       992



In [68]:
# ── Confusion matrix ─────────────────────────────────────────
cm = confusion_matrix(test_true, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — {MODEL_NAME.split('/')[-1]}")
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("bert_confusion_matrix.png", dpi=150)
plt.close()
print("Saved: bert_confusion_matrix.png")

Saved: bert_confusion_matrix.png


In [69]:
# SECTION 11: TRAINING CURVES
# ==============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend()
axes[0].set_xlabel("Epoch")

axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"],   label="Val")
axes[1].set_title("Accuracy"); axes[1].legend()
axes[1].set_xlabel("Epoch")

axes[2].plot(history["val_f1"], color="green", marker="o")
axes[2].set_title("Val Macro-F1"); axes[2].set_xlabel("Epoch")
axes[2].axhline(y=best_val_f1, color="red", linestyle="--",
                label=f"Best: {best_val_f1:.4f}")
axes[2].legend()

plt.suptitle(f"BERT Training History — {MODEL_NAME.split('/')[-1]}", fontsize=13)
plt.tight_layout()
plt.savefig("bert_training_curves.png", dpi=150)
plt.close()
print("Saved: bert_training_curves.png")

Saved: bert_training_curves.png


In [70]:
# SECTION 12: EXPLAINABILITY WITH SHAP
# ==============================================================

try:
    import shap

    print("\nRunning SHAP explainability (on 50 test samples)...")

    def bert_predict_proba(texts):
        """Wrapper for SHAP: returns probability matrix."""
        best_model.eval()
        enc = tokenizer(
            list(texts),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()
               if k in ["input_ids", "attention_mask", "token_type_ids"]}
        with torch.no_grad():
            logits = best_model(**enc).logits
        return torch.softmax(logits, dim=-1).cpu().numpy()

    explainer  = shap.Explainer(bert_predict_proba, tokenizer)
    shap_texts = test_texts[:50]
    shap_vals  = explainer(shap_texts)

    shap.plots.bar(shap_vals[:, :, 0], max_display=15, show=False)
    plt.tight_layout()
    plt.savefig("shap_feature_importance.png", dpi=130)
    plt.close()
    print("Saved: shap_feature_importance.png")

except ImportError:
    print("SHAP not installed — skipping. Install with: pip install shap")
except Exception as e:
    print(f"SHAP skipped: {e}")


Running SHAP explainability (on 50 test samples)...


PartitionExplainer explainer: 51it [51:57, 62.35s/it]                        


Saved: shap_feature_importance.png


In [71]:
# SECTION 13: INFERENCE — REAL-TIME PREDICTION
# ==============================================================

def predict_with_bert(text: str, model=best_model, tok=tokenizer,
                      label_enc=le, device=DEVICE, max_len=MAX_LEN) -> dict:
    """
    Predict mental health category from raw text using fine-tuned BERT.
    Returns prediction, confidence, and all class probabilities.
    """
    model.eval()
    cleaned = basic_clean(text)
    enc = tok(
        cleaned,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()
           if k in ["input_ids", "attention_mask", "token_type_ids"]}

    with torch.no_grad():
        logits = model(**enc).logits
        probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

    pred_idx = probs.argmax()
    sorted_probs = sorted(enumerate(probs), key=lambda x: x[1], reverse=True)

    return {
        "text"        : text[:100],
        "prediction"  : label_enc.classes_[pred_idx],
        "confidence"  : f"{probs[pred_idx]*100:.1f}%",
        "top_3"       : [(label_enc.classes_[i], f"{p*100:.1f}%")
                         for i, p in sorted_probs[:3]],
    }

In [72]:
# ── Demo predictions ─────────────────────────────────────────
demo_texts = [
    "I've been feeling extremely hopeless and I cry myself to sleep every night",
    "My racing thoughts won't stop, I'm terrified all the time for no reason",
    "Had an amazing workout this morning, feeling energized and positive!",
    "I've been planning how to end my life, I've written a note already",
    "My energy goes from extreme highs where I need no sleep to crushing lows",
    "I feel detached from myself sometimes, like I'm watching my life from outside",
]

print("\n\n── BERT INFERENCE DEMO ──────────────────────────────────")
print(f"  Model: {MODEL_NAME}")
print("="*65)
for txt in demo_texts:
    result = predict_with_bert(txt)
    print(f"\n  INPUT      : {result['text'][:70]}...")
    print(f"  PREDICTION : {result['prediction']}  ({result['confidence']})")
    print(f"  TOP-3      : {result['top_3']}")



── BERT INFERENCE DEMO ──────────────────────────────────
  Model: bert-base-uncased

  INPUT      : I've been feeling extremely hopeless and I cry myself to sleep every n...
  PREDICTION : Depression  (80.2%)
  TOP-3      : [('Depression', '80.2%'), ('Suicidal', '12.3%'), ('Anxiety', '6.7%')]

  INPUT      : My racing thoughts won't stop, I'm terrified all the time for no reaso...
  PREDICTION : Anxiety  (97.0%)
  TOP-3      : [('Anxiety', '97.0%'), ('Normal', '2.9%'), ('Depression', '0.1%')]

  INPUT      : Had an amazing workout this morning, feeling energized and positive!...
  PREDICTION : Anxiety  (80.7%)
  TOP-3      : [('Anxiety', '80.7%'), ('Normal', '18.3%'), ('Depression', '0.9%')]

  INPUT      : I've been planning how to end my life, I've written a note already...
  PREDICTION : Suicidal  (86.1%)
  TOP-3      : [('Suicidal', '86.1%'), ('Depression', '9.8%'), ('Normal', '3.7%')]

  INPUT      : My energy goes from extreme highs where I need no sleep to crushing lo...
  PR

In [73]:
# SECTION 14: FINAL SUMMARY
# ==============================================================

print("\n\n" + "="*65)
print("  BERT PIPELINE COMPLETE")
print("="*65)
print(f"  Model          : {MODEL_NAME}")
print(f"  Best Val F1    : {best_val_f1:.4f}")
print(f"  Test Accuracy  : {test_acc:.4f}")
print(f"  Test Macro-F1  : {test_f1:.4f}")
print(f"  Saved to       : {best_model_path}/")
print("\n  Saved artifacts:")
print("    bert_confusion_matrix.png")
print("    bert_training_curves.png")
print("    shap_feature_importance.png  (if SHAP installed)")
print("\n  To switch models, change MODEL_NAME at the top to:")
print("    'roberta-base'")
print("    'mental/mental-bert-base-uncased'   ← best for this dataset")
print("="*65)



  BERT PIPELINE COMPLETE
  Model          : bert-base-uncased
  Best Val F1    : 0.8238
  Test Accuracy  : 0.8538
  Test Macro-F1  : 0.8521
  Saved to       : ./bert_checkpoints\best_model/

  Saved artifacts:
    bert_confusion_matrix.png
    bert_training_curves.png
    shap_feature_importance.png  (if SHAP installed)

  To switch models, change MODEL_NAME at the top to:
    'roberta-base'
    'mental/mental-bert-base-uncased'   ← best for this dataset
